# SolarSystemLander Mission Analysis

Compare Study Series **2A (8D, no weather report)** and **2B (11D, weather report)**.

Goals:

- understand where each lander succeeds and fails,
- distinguish quality gains from training-effort gains,
- identify promising regions for the next hyperparameter search space.

The robust Study metrics include additional seeds. Per-world scores are stored only for the original Optuna trials, not for those additional robustness runs.


In [2]:
from pathlib import Path
import json
import math
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DB_DIR = Path("../tmp")
DATABASES = {
    "2A (8D)": DB_DIR / "solar_system_lander_8d.db",
    "2B (11D)": DB_DIR / "solar_system_lander_11d.db",
}
WORLDS = ["moon", "mercury", "mars", "earth", "venus"]
PARAMETERS = [
    "learning_rate",
    "batch_size",
    "eps_end",
    "eps_decay",
    "learning_starts",
    "optimize_every",
    "replay_memory_capacity",
    "num_episodes",
]

missing = [str(path) for path in DATABASES.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing databases: {missing}")


## Load Optuna SQLite data

The loader uses SQLite directly, so analysis does not depend on the Optuna Python package. Categorical parameter values are decoded from Optuna's stored distributions.


In [3]:
def _json_attributes(connection, table, id_column, object_id):
    rows = connection.execute(
        f'SELECT "key", value_json FROM {table} WHERE {id_column} = ?',
        (object_id,),
    )
    return {key: json.loads(value) for key, value in rows}


def _decode_param(value, distribution_json):
    distribution = json.loads(distribution_json)
    if distribution["name"] == "CategoricalDistribution":
        return distribution["attributes"]["choices"][int(value)]
    if distribution["name"] == "IntDistribution":
        return int(value)
    return value


def load_database(series, path):
    studies = []
    trials = []
    with sqlite3.connect(path) as connection:
        connection.row_factory = sqlite3.Row
        inherited_params = {}
        for study_index, study in enumerate(
            connection.execute("SELECT * FROM studies ORDER BY study_id")
        ):
            study_attrs = _json_attributes(
                connection, "study_user_attributes", "study_id", study["study_id"]
            )
            inherited_params.update(study_attrs.get("baseline_params", {}))
            studies.append({
                "series": series,
                "study": study["study_name"].split("_", 1)[0].upper(),
                "study_name": study["study_name"],
                "study_index": study_index,
                "qe_score": study_attrs.get("robust_best_objective_score"),
                "gym_score": study_attrs.get("robust_best_gym_score"),
                "training_effort": study_attrs.get("robust_best_training_effort"),
                "robust_params": study_attrs.get("robust_best_params", {}),
            })

            trial_rows = connection.execute(
                """
                SELECT t.trial_id, t.number, v.value
                FROM trials t
                JOIN trial_values v ON v.trial_id = t.trial_id AND v.objective = 0
                WHERE t.study_id = ? AND t.state = 'COMPLETE'
                ORDER BY t.number
                """,
                (study["study_id"],),
            )
            for trial in trial_rows:
                attrs = _json_attributes(
                    connection, "trial_user_attributes", "trial_id", trial["trial_id"]
                )
                params = {
                    row["param_name"]: _decode_param(
                        row["param_value"], row["distribution_json"]
                    )
                    for row in connection.execute(
                        "SELECT * FROM trial_params WHERE trial_id = ?",
                        (trial["trial_id"],),
                    )
                }
                full_params = inherited_params | params
                trials.append({
                    "series": series,
                    "study": study["study_name"].split("_", 1)[0].upper(),
                    "study_name": study["study_name"],
                    "study_index": study_index,
                    "trial": trial["number"],
                    "qe_score": trial["value"],
                    "gym_score": attrs.get("gym_score"),
                    "training_effort": attrs.get("training_effort"),
                    **attrs.get("gym_scores", {}),
                    **full_params,
                })
            inherited_params.update(study_attrs.get("robust_best_params", {}))
    return studies, trials


study_rows = []
trial_rows = []
for series, path in DATABASES.items():
    studies, trials = load_database(series, path)
    study_rows.extend(studies)
    trial_rows.extend(trials)

studies = pd.DataFrame(study_rows)
trials = pd.DataFrame(trial_rows)

studies[["series", "study", "gym_score", "qe_score", "training_effort"]]


,series,study,gym_score,qe_score,training_effort
0,2A (8D),S0,-20.558651,-3.087821,1.000000
1,2A (8D),S1,35.909499,-2.143829,0.488542
2,2A (8D),S2,79.541797,-1.496920,0.368349
3,2A (8D),S3,52.439787,-1.891564,0.419071
4,2A (8D),S4,111.657420,-1.005922,0.230420
5,2B (11D),S0,19.070257,-2.533016,1.000000
6,2B (11D),S1,92.456168,-1.400070,0.648188
7,2B (11D),S2,64.086551,-1.733701,0.436377
8,2B (11D),S3,42.787933,-2.036696,0.452424
9,2B (11D),S4,61.520545,-1.902872,0.880531


## 1. Study Series

Robust results after each Study. This is the mission-level comparison and includes the additional robustness seeds.


In [4]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Gym score", "QE score", "Training effort relative to S0"),
)
colors = {"2A (8D)": "#1f77b4", "2B (11D)": "#ff7f0e"}
for series, group in studies.groupby("series", sort=False):
    group = group.sort_values("study_index")
    for column, col in [("gym_score", 1), ("qe_score", 2), ("training_effort", 3)]:
        fig.add_trace(
            go.Scatter(
                x=group["study"],
                y=group[column],
                mode="lines+markers+text",
                text=group[column].round(2),
                textposition="top center",
                name=series,
                legendgroup=series,
                showlegend=col == 1,
                line=dict(color=colors[series]),
            ),
            row=1, col=col,
        )
fig.add_hline(y=200, line_dash="dash", line_color="gray", row=1, col=1)
fig.update_layout(height=430, width=1200, title="Robust Study Series Results")
fig.show()


## 2. Trial landscape

Every point is one original Optuna trial. Hover shows its Study and hyperparameters. This reveals whether success is broad or depends on rare parameter combinations.


In [5]:
hover_parameters = [parameter for parameter in PARAMETERS if parameter in trials.columns]
fig = px.scatter(
    trials,
    x="training_effort",
    y="gym_score",
    color="series",
    symbol="study",
    hover_data=["trial", "qe_score", *hover_parameters],
    color_discrete_map=colors,
    title="Quality versus Training Effort for All Trials",
    labels={"training_effort": "Training effort relative to S0", "gym_score": "Gym score"},
)
fig.add_hline(y=200, line_dash="dash", line_color="gray")
fig.add_hline(y=250, line_dash="dot", line_color="gray")
fig.update_layout(height=600, width=1100)
fig.show()


## 3. Performance by world

Mean raw trial score for each world and Study. These values diagnose where learning succeeds or breaks down; they are not robustness-seed averages.


In [7]:
world_scores = trials.melt(
    id_vars=["series", "study", "study_index", "trial"],
    value_vars=WORLDS,
    var_name="world",
    value_name="world_score",
)
world_summary = (
    world_scores.groupby(["series", "study", "study_index", "world"], as_index=False)
    ["world_score"].mean()
    .sort_values(["study_index", "world"])
)
fig = px.bar(
    world_summary,
    x="world",
    y="world_score",
    color="series",
    facet_col="study",
    barmode="group",
    color_discrete_map=colors,
    category_orders={"study": ["S0", "S1", "S2", "S3", "S4"], "world": WORLDS},
    title="Mean World Score of Original Trials",
)
fig.update_layout(height=470, width=1400)
fig.show()


## 4. Best raw trial by world

This shows the world profile of the highest-QE original trial in every Study. A strong average can still hide a failed planet.


In [8]:
best_indices = trials.groupby(["series", "study"])["qe_score"].idxmax()
best_trials = trials.loc[best_indices].copy()
best_worlds = best_trials.melt(
    id_vars=["series", "study", "trial", "qe_score", "gym_score", "training_effort"],
    value_vars=WORLDS,
    var_name="world",
    value_name="world_score",
)
fig = px.line(
    best_worlds,
    x="world",
    y="world_score",
    color="series",
    facet_col="study",
    markers=True,
    hover_data=["trial", "qe_score", "gym_score", "training_effort"],
    color_discrete_map=colors,
    category_orders={"study": ["S0", "S1", "S2", "S3", "S4"], "world": WORLDS},
    title="World Profile of Each Study's Best Raw Trial",
)
fig.add_hline(y=200, line_dash="dash", line_color="gray")
fig.update_layout(height=470, width=1400)
fig.show()


## 5. S4 joint-search parameter landscape

S4 is the most useful evidence for a new joint search because it varies several parameters together. Parallel coordinates expose combinations rather than misleading one-parameter effects.


In [9]:
s4 = trials[trials["study"] == "S4"].copy()
dimensions = [parameter for parameter in PARAMETERS if parameter in s4.columns]
for series, group in s4.groupby("series", sort=False):
    figure = px.parallel_coordinates(
        group,
        dimensions=[*dimensions, "gym_score", "training_effort", "qe_score"],
        color="qe_score",
        color_continuous_scale=px.colors.diverging.Tealrose,
        title=f"{series}: S4 Joint Search",
    )
    figure.update_layout(height=560, width=1300)
    figure.show()


## 6. Candidate ranges for the next search

The table summarizes the top 20% of S4 trials by QE, with at least five trials per series. Inherited fixed parameters are included. It is evidence for a focused follow-up search?not a final causal conclusion. Historical S4 was centered on the old sequential winners, so copy promising regions rather than its bounds blindly. The new Study Series should include the incumbent explicitly and robustly re-evaluate winners.


In [11]:
CATEGORICAL_PARAMETERS = {
    "batch_size",
    "learning_starts",
    "optimize_every",
    "replay_memory_capacity",
    "num_episodes",
}

def top_fraction(group, fraction=0.20, minimum=5):
    return group.nlargest(max(minimum, math.ceil(len(group) * fraction)), "qe_score")

top_s4 = pd.concat(
    [top_fraction(group) for _, group in s4.groupby("series")],
    ignore_index=True,
)

range_rows = []
for series, group in top_s4.groupby("series"):
    for parameter in dimensions:
        values = group[parameter].dropna()
        if values.empty:
            continue
        if parameter in CATEGORICAL_PARAMETERS:
            recommendation = ", ".join(
                f"{value:g} ({count}x)"
                for value, count in values.value_counts().items()
            )
        else:
            recommendation = (
                f"{values.min():.6g} .. {values.max():.6g} "
                f"(median {values.median():.6g})"
            )
        range_rows.append({
            "series": series,
            "parameter": parameter,
            "top-trial region": recommendation,
        })

candidate_ranges = pd.DataFrame(range_rows)
display(candidate_ranges.pivot(
    index="parameter", columns="series", values="top-trial region"
).reindex(dimensions))

top_columns = [
    "series", "trial", "qe_score", "gym_score", "training_effort",
    *dimensions,
]
top_s4[top_columns].sort_values(["series", "qe_score"], ascending=[True, False])


series,2A (8D),2B (11D)
parameter,,
learning_rate,0.00196386 .. 0.00232865 (median 0.00223019),0.000939068 .. 0.00154888 (median 0.00134639)
batch_size,"512 (3x), 1024 (2x)",1024 (5x)
eps_end,0.0397103 .. 0.0492957 (median 0.0418424),0.0455388 .. 0.0489798 (median 0.0478626)
eps_decay,33269 .. 52496 (median 49723),23775 .. 28745 (median 24995)
learning_starts,"1000 (3x), 2500 (2x)","1000 (4x), 2500 (1x)"
optimize_every,4 (5x),"2 (4x), 8 (1x)"
replay_memory_capacity,400000 (5x),200000 (5x)
num_episodes,500 (5x),500 (5x)


,series,trial,qe_score,gym_score,training_effort,learning_rate,batch_size,eps_end,eps_decay,learning_starts,optimize_every,replay_memory_capacity,num_episodes
0,2A (8D),1,-0.011359,181.659841,0.181988,0.002329,512,0.041842,49723,2500,4,400000,500
1,2A (8D),22,-0.835365,124.246881,0.249403,0.001979,1024,0.045224,52496,1000,4,400000,500
2,2A (8D),11,-1.412300,81.585133,0.181641,0.002230,512,0.040612,51472,1000,4,400000,500
3,2A (8D),18,-1.779669,57.407517,0.277914,0.001964,1024,0.039710,43524,1000,4,400000,500
4,2A (8D),12,-1.899517,49.877662,0.326014,0.002297,512,0.049296,33269,2500,4,400000,500
5,2B (11D),23,-1.785267,70.397052,0.902753,0.000939,1024,0.047863,25376,1000,2,200000,500
6,2B (11D),4,-1.848827,55.784430,0.432695,0.001346,1024,0.045777,23775,2500,8,200000,500
7,2B (11D),14,-1.924302,59.439768,0.854863,0.001549,1024,0.048980,23905,1000,2,200000,500
8,2B (11D),24,-2.034388,48.798744,0.725233,0.001447,1024,0.048317,24995,1000,2,200000,500
9,2B (11D),1,-2.195792,38.260995,0.771487,0.001012,1024,0.045539,28745,1000,2,200000,500


## Interpretation checklist for the next mission

Before defining the next search spaces:

1. Find parameters that recur across several strong trials, not only the single winner.
2. Check whether Gym gains are broad across worlds or bought by sacrificing one planet.
3. Treat 2A and 2B separately if their promising regions differ materially.
4. Keep the incumbent in every Study so a narrower search cannot lower the mission result.
5. Robustly re-evaluate the most promising configurations with additional seeds.
6. For 2B, investigate observation quality separately from HPO: nominal weather inputs may not describe instantaneous gusts.
